# 15장. 레이블이 없는 자료: k-평균 군집과 주성분 분석

예측할 목표가 없는 자료에서 비슷한 것들의 집단을 찾고 특성의 수를 줄이는 방법, 그리고 찾은 것이 맞는지 확인하는 방법.

**준비:** CSV 여섯 개(전력수요 셋, 기상 둘, 서울 지하철 시간대별 승하차 하나)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 3분쯤 걸린다.

## 15.1 k-평균 군집

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in csvs if first_line(f).startswith("날짜,1시")]).dropna(subset=["날짜"])
wide["날짜"] = pd.to_datetime(wide["날짜"])
wide = wide.set_index("날짜").sort_index()
GW = wide.to_numpy() / 1000                          # 행이 하루, 열이 1시부터 24시까지
shape = GW / GW.mean(axis=1, keepdims=True)          # 그날의 평균으로 나눈다. 수준은 없어지고 하루 안의 모양만 남는다
dates = wide.index

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01", "2023-05-05", "2023-05-27", "2023-05-29",
    "2023-06-06", "2023-08-15", "2023-09-28", "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2024-01-01", "2024-02-09", "2024-02-10", "2024-02-11", "2024-02-12", "2024-03-01", "2024-04-10", "2024-05-05", "2024-05-06", "2024-05-15",
    "2024-06-06", "2024-08-15", "2024-09-16", "2024-09-17", "2024-09-18", "2024-10-01", "2024-10-03", "2024-10-09", "2024-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01", "2025-03-03", "2025-05-05", "2025-05-06",
    "2025-06-03", "2025-06-06", "2025-08-15", "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09", "2025-12-25"])
kind = np.where(dates.isin(HOLIDAYS), "공휴일", np.where(dates.dayofweek < 5, "평일", np.where(dates.dayofweek == 5, "토요일", "일요일")))
print(GW.shape, dates.min().date(), dates.max().date(), pd.Series(kind).value_counts().to_dict())
for day in ["2025-01-15", "2025-04-13"]:
    i = dates.get_loc(day)
    print(day, kind[i], round(float(GW[i].mean()), 1), shape[i, [3, 9, 12, 18]].round(2).tolist())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 15-1
HOURS24 = np.arange(1, 25)
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.5))
for day, color in [("2025-01-15", NAVY), ("2025-04-13", BRICK)]:
    i = dates.get_loc(day)
    label = f"{day} ({kind[i]}), 평균 {GW[i].mean():.1f}GW"
    axes[0].plot(HOURS24, GW[i], color=color, lw=1.8, marker="o", ms=2.5, label=label)
    axes[1].plot(HOURS24, shape[i], color=color, lw=1.8, marker="o", ms=2.5, label=day)
axes[1].axhline(1, color=GRAY, lw=0.9, ls=":", label="그날의 평균(= 1)")
axes[0].set_ylabel("전력수요(GW)", fontsize=8)
axes[1].set_ylabel("그날의 평균에 대한 비", fontsize=8)
axes[0].set_ylim(40, 92)
axes[0].legend(frameon=False, fontsize=7, loc="upper left")
axes[1].legend(frameon=False, fontsize=7, loc="lower right")
for ax, title in zip(axes, ["시간별 수요", "하루의 모양(평균으로 나눈 값)"]):
    ax.set_xticks([1, 4, 7, 10, 13, 16, 19, 22, 24])
    ax.set_xlabel("시(1시는 0~1시의 수요)", fontsize=8)
    frame(ax, title)
fig.tight_layout()
fig.savefig("fig_15_1.png", dpi=300)
plt.show()

In [ ]:
def kmeans(X, k, seed, laps=100, trace=None):
    """줄들을 k개 군집으로 나눈다. (군집 번호, 군집의 중심, 줄마다 자기 중심까지의 거리 제곱의 합)을 돌려준다."""
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)]                           # 무작위로 고른 k줄을 처음 중심으로 한다
    for _ in range(laps):
        far = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)           # 줄마다 중심 k개까지의 거리 제곱
        label = far.argmin(axis=1)                                               # 배정: 가장 가까운 중심의 군집에 넣는다
        if trace is not None:
            trace.append((centers.copy(), label.copy(), float(far.min(axis=1).sum())))
        moved = np.array([X[label == j].mean(axis=0) if (label == j).any() else centers[j] for j in range(k)])
        if np.allclose(moved, centers):                                          # 중심이 더 움직이지 않으면 멈춘다
            break
        centers = moved                                                          # 갱신: 중심을 그 군집에 속한 줄들의 평균으로 옮긴다
    return label, centers, float(far.min(axis=1).sum())

steps = []
label, centers, total = kmeans(shape, 2, seed=3, trace=steps)
print(len(steps), [round(s[2], 2) for s in steps])
print(np.bincount(steps[0][1]).tolist(), np.bincount(label).tolist())

In [ ]:
# 그림 15-2
show = [0, 1, 2, len(steps) - 1]
fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.2), sharex=True, sharey=True)
for ax, s in zip(axes.ravel(), show):
    cen, lab, tot = steps[s]
    for j, color in enumerate([NAVY, BRICK]):
        ax.scatter(shape[lab == j, 3], shape[lab == j, 12], s=5, color=color, alpha=0.35, lw=0)
        ax.scatter(cen[j, 3], cen[j, 12], s=110, marker="X", color=color, edgecolor="white", lw=1.2, zorder=5)
    if s > 0:
        for j in range(2):
            trail = np.array([steps[q][0][j][[3, 12]] for q in range(s + 1)])     # 중심이 지금까지 지나온 위치
            ax.plot(trail[:, 0], trail[:, 1], color="black", lw=1.0, marker="o", ms=2.2, zorder=4)
    title = ("처음: 무작위로 고른 두 줄이 중심" if s == 0 else f"{s}번 갱신한 뒤" if s < len(steps) - 1 else f"{s}번 갱신한 뒤: 더 움직이지 않는다")
    frame(ax, title, grid="both")
    ax.text(0.03, 0.04, f"거리 제곱의 합 {tot:.2f}\n군집 크기 {(lab == 0).sum()}일 / {(lab == 1).sum()}일" + ("\n검은 선: 중심이 지나온 위치" if s > 0 else "\n× 표시: 군집의 중심"), transform=ax.transAxes, fontsize=7.2, color=NAVY, bbox=BOX, va="bottom", linespacing=1.5)
for ax in axes[1]:
    ax.set_xlabel("4시의 값(평균에 대한 비)", fontsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel("13시의 값(평균에 대한 비)", fontsize=8)
[a_.tick_params(labelbottom=True, labelleft=True) for a_ in fig.axes]
fig.tight_layout()
fig.savefig("fig_15_2.png", dpi=300)
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

def best_of(X, k, tries=10):
    """처음 중심을 바꿔 가며 여러 번 실행해서 거리 제곱의 합이 가장 작은 결과를 고른다. 군집 번호는 큰 군집부터 0, 1, 2로 다시 매긴다."""
    runs = [kmeans(X, k, seed) for seed in range(tries)]
    label, centers, total = min(runs, key=lambda r: r[2])
    order = np.argsort(-np.bincount(label, minlength=k))
    return np.argsort(order)[label], centers[order], total, runs

two, centers2, total2, runs2 = best_of(shape, 2)
whole = float(((shape - shape.mean(axis=0)) ** 2).sum())                         # 군집이 하나일 때의 거리 제곱의 합
print(round(whole, 2), round(total2, 2), round(1 - total2 / whole, 3), sorted({round(r[2], 2) for r in runs2}))
theirs = KMeans(2, n_init=10, random_state=0).fit(shape)
print(round(float(theirs.inertia_), 2), round(adjusted_rand_score(two, theirs.labels_), 3))
mix2 = pd.crosstab(pd.Series(two, name="군집"), pd.Series(kind, name=""))[["평일", "토요일", "일요일", "공휴일"]]
print(mix2.to_string())
print(centers2[:, [3, 9, 12, 18, 21]].round(3).tolist())

In [ ]:
# 그림 15-3
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 3.6), gridspec_kw={"width_ratios": [1.5, 1]})
for j, color in enumerate([NAVY, BRICK]):
    members = shape[two == j]
    ax.fill_between(HOURS24, np.quantile(members, 0.1, axis=0), np.quantile(members, 0.9, axis=0), color=color, alpha=0.13, lw=0)
    ax.plot(HOURS24, centers2[j], color=color, lw=2, marker="o", ms=2.5, label=f"군집 {j}의 중심({(two == j).sum()}일)")
ax.axhline(1, color=GRAY, lw=0.8, ls=":")
ax.set_xticks([1, 4, 7, 10, 13, 16, 19, 22, 24])
ax.set_xlabel("시", fontsize=8)
ax.set_ylabel("그날의 평균에 대한 비", fontsize=8)
ax.legend(frameon=False, fontsize=7.5, loc="lower right")
frame(ax, "두 군집의 중심. 띠는 군집 안 10~90% 범위")
kinds = ["평일", "토요일", "일요일", "공휴일"]
share0 = [mix2.loc[0, c] / mix2[c].sum() * 100 for c in kinds]
bx.bar(kinds, share0, color=NAVY, width=0.6, label="군집 0")
bx.bar(kinds, [100 - v for v in share0], bottom=share0, color=BRICK, width=0.6, label="군집 1")
for x, v, c in zip(range(4), share0, kinds):
    bx.text(x, v / 2, f"{mix2.loc[0, c]}일", fontsize=7.5, color="white", ha="center", va="center")
    bx.text(x, v + (100 - v) / 2, f"{mix2.loc[1, c]}일", fontsize=7.5, color="white", ha="center", va="center")
bx.set_ylabel("그 종류의 날 가운데 비율(%)", fontsize=8)
bx.set_ylim(0, 100)
frame(bx, "달력의 날 종류별 구성")
fig.tight_layout()
fig.savefig("fig_15_3.png", dpi=300)
plt.show()

In [ ]:
season = pd.cut(dates.month, [0, 2, 5, 8, 11, 12], labels=["1~2월", "3~5월", "6~8월", "9~11월", "12월"])
plain2, plain_centers, _, _ = best_of(GW, 2)                                    # 평균으로 나누지 않고 GW 그대로 군집을 찾는다
print(pd.crosstab(pd.Series(plain2, name="군집"), pd.Series(season, name="")).to_string())
print(pd.crosstab(pd.Series(plain2, name="군집"), pd.Series(kind, name=""))[["평일", "토요일", "일요일", "공휴일"]].to_string())
print(plain_centers.mean(axis=1).round(1).tolist(), round(adjusted_rand_score(plain2, two), 2))
level = GW.mean(axis=1)
cut = (level[plain2 == 0].max() + level[plain2 == 1].min()) / 2
print(round(float(level[plain2 == 0].max()), 1), round(float(level[plain2 == 1].min()), 1), round(float(((level > cut) == (plain2 == 1)).mean()), 3))

In [ ]:
# 그림 15-4
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.5), sharey=True)
for ax, lab, title in [(axes[0], plain2, "GW 그대로 찾은 두 군집"), (axes[1], two, "모양으로 찾은 두 군집")]:
    for j, color in enumerate([NAVY, BRICK]):
        ax.scatter(dates[lab == j], level[lab == j], s=5, color=color, alpha=0.7, lw=0, label=f"군집 {j} ({(lab == j).sum()}일)")
    ax.legend(frameon=False, fontsize=7.5, loc="upper center", ncol=2, markerscale=2.5)
    ax.set_ylim(42, 96)
    ax.set_xticks(pd.to_datetime(["2023-01-01", "2023-07-01", "2024-01-01", "2024-07-01", "2025-01-01", "2025-07-01", "2026-01-01"]),
                  ["23.1", "23.7", "24.1", "24.7", "25.1", "25.7", "26.1"], fontsize=7.5)
    ax.set_xlabel("날짜(연.월)", fontsize=8)
    frame(ax, title)
axes[0].axhline(cut, color="black", lw=0.9, ls="--")
axes[0].text(0.98, 0.03, f"검은 점선: 일평균 {cut:.1f}GW", transform=axes[0].transAxes, fontsize=7, color=NAVY, bbox=BOX, ha="right")
axes[0].set_ylabel("그날의 평균 수요(GW)", fontsize=8)
[a_.tick_params(labelbottom=True, labelleft=True) for a_ in fig.axes]
fig.tight_layout()
fig.savefig("fig_15_4.png", dpi=300)
plt.show()

## 15.2 군집의 수와 군집이 실제로 있는지 확인하기

In [ ]:
def silhouette(X, label):
    """줄마다 (가장 가까운 다른 군집까지의 평균 거리 - 자기 군집 안의 평균 거리)를 둘 가운데 큰 값으로 나눈다. 그 평균이 실루엣 계수다."""
    apart = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))          # 모든 줄 사이의 거리
    marks = []
    for i in range(len(X)):
        mine = label == label[i]
        inside = apart[i, mine].sum() / max(mine.sum() - 1, 1)
        outside = min(apart[i, label == j].mean() for j in np.unique(label) if j != label[i])
        marks.append((outside - inside) / max(inside, outside))
    return float(np.mean(marks))

rows = []
for k in [2, 3, 4, 5, 6, 8, 10]:
    label, _, total, runs = best_of(shape, k)
    same = [adjusted_rand_score(runs[0][0], r[0]) for r in runs[1:]]             # 첫 번째 실행과 나머지 아홉 번의 결과가 얼마나 같은가
    rows.append((k, round(total, 2), round(1 - total / whole, 3), round(silhouette(shape, label), 3), round(float(np.min(same)), 2), round(float(np.mean(same)), 2)))
pick = pd.DataFrame(rows, columns=["k", "제곱합", "줄어든 비율", "실루엣", "일치(최저)", "일치(평균)"])
print(pick.to_string(index=False))

In [ ]:
# 그림 15-5
fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0))
ks = pick["k"].to_numpy()
panels = [("제곱합", "군집 내 거리 제곱의 합", NAVY), ("실루엣", "실루엣 계수", TEAL), ("일치(최저)", "실행 사이의 일치(조정 랜드 지수)", BRICK)]
for ax, (col, title, color) in zip(axes, panels):
    ax.plot(ks, pick[col], color=color, lw=1.8, marker="o", ms=4, label="가장 낮은 값" if col == "일치(최저)" else None)
    for x, v in zip(ks, pick[col]):
        off_ = {("실루엣", 6): (-4, -11), ("일치(최저)", 2): (-3, 7), ("일치(최저)", 3): (7, 4), ("일치(최저)", 5): (-9, -11), ("일치(최저)", 6): (3, 6)}.get((col, int(x)), (4, 5))
        ax.annotate(f"{v:.2f}" if col != "제곱합" else f"{v:.1f}", (x, v), xytext=off_, textcoords="offset points", fontsize=6.5, color=color)
    ax.set_xticks(ks)
    ax.set_xlim(1.4, 11.6)
    ax.set_xlabel("군집의 수 k", fontsize=8)
    frame(ax, title)
axes[2].plot(ks, pick["일치(평균)"], color=GRAY, lw=1.4, marker="s", ms=3, ls="--", label="평균")
axes[2].legend(frameon=False, fontsize=7, loc="lower left")
axes[0].set_ylim(10, 54)
axes[1].set_ylim(0.25, 0.41)
axes[2].set_ylim(0.3, 1.15)
fig.tight_layout()
fig.savefig("fig_15_5.png", dpi=300)
plt.show()

In [ ]:
rng = np.random.default_rng(0)
stirred = np.column_stack([rng.permutation(shape[:, j]) for j in range(24)])     # 시간마다 날을 따로 섞는다. 열의 분포는 그대로이고 하루의 모양이 없어진다
flat = rng.random((len(shape), 2))                                               # 정사각형 안에 고르게 뿌린 점 1,096개
found = {}
for name, X in [("시간마다 섞은 자료", stirred), ("고르게 뿌린 점", flat)]:
    out = []
    for k in (2, 3, 4):
        label, cen, total, runs = best_of(X, k)
        base = float(((X - X.mean(axis=0)) ** 2).sum())
        out.append((k, round(1 - total / base, 3), round(silhouette(X, label), 3), round(float(np.min([adjusted_rand_score(runs[0][0], r[0]) for r in runs[1:]])), 2)))
        found[(name, k)] = (label, cen)
    print(name, out)

In [ ]:
# 그림 15-6
lab4, cen4 = found[("고르게 뿌린 점", 4)]
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 3.7))
ax.scatter(flat[:, 0], flat[:, 1], s=6, color=GRAY, lw=0)
frame(ax, "고르게 뿌린 점 1,096개", grid="both")
for j, color in enumerate([NAVY, BRICK, TEAL, "#8a6d3b"]):
    bx.scatter(flat[lab4 == j, 0], flat[lab4 == j, 1], s=6, color=color, lw=0)
    bx.scatter(cen4[j, 0], cen4[j, 1], s=120, marker="X", color=color, edgecolor="white", lw=1.3, zorder=5)
frame(bx, f"k-평균이 찾은 네 군집. 실루엣 {silhouette(flat, lab4):.3f}", grid="both")
for a in (ax, bx):
    a.set_xlim(-0.02, 1.02); a.set_ylim(-0.02, 1.02)
    a.set_xlabel("첫 번째 좌표", fontsize=8)
    a.set_aspect("equal")
ax.set_ylabel("두 번째 좌표", fontsize=8)
fig.tight_layout()
fig.savefig("fig_15_6.png", dpi=300)
plt.show()

In [ ]:
four, centers4, _, _ = best_of(shape, 4)
NAMES4 = ["가", "나", "다", "라"]
named = pd.Series(np.array(NAMES4)[four], name="군집")
mix4 = pd.crosstab(named, pd.Series(kind, name=""))[["평일", "토요일", "일요일", "공휴일"]]
print(mix4.to_string())
sea4 = pd.crosstab(named, pd.Series(season, name=""))
print(sea4.to_string())
print(pd.DataFrame(centers4[:, [3, 9, 12, 18, 21]], index=NAMES4, columns=["4시", "10시", "13시", "19시", "22시"]).round(3).to_string())

In [ ]:
# 그림 15-7
COL4 = [NAVY, TEAL, "#8a6d3b", BRICK]
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 3.7), gridspec_kw={"width_ratios": [1.45, 1]})
for j, color in enumerate(COL4):
    ax.plot(HOURS24, centers4[j], color=color, lw=1.9, marker="o", ms=2.3, label=f"{NAMES4[j]} ({(four == j).sum()}일)")
ax.axhline(1, color=GRAY, lw=0.8, ls=":")
ax.set_xticks([1, 4, 7, 10, 13, 16, 19, 22, 24])
ax.set_xlabel("시", fontsize=8)
ax.set_ylabel("그날의 평균에 대한 비", fontsize=8)
ax.legend(frameon=False, fontsize=7.5, loc="lower right", ncol=2)
frame(ax, "네 군집의 중심")
bottom = np.zeros(4)
for j, color in enumerate(COL4):
    vals = np.array([mix4.loc[NAMES4[j], c] / mix4[c].sum() * 100 for c in kinds])
    bx.bar(kinds, vals, bottom=bottom, color=color, width=0.62, label=NAMES4[j])
    for x, (v, b) in enumerate(zip(vals, bottom)):
        if v >= 9:
            bx.text(x, b + v / 2, f"{mix4.loc[NAMES4[j], kinds[x]]}", fontsize=7, color="white", ha="center", va="center")
    bottom += vals
bx.set_ylim(0, 100)
bx.set_ylabel("그 종류의 날 가운데 비율(%). 숫자는 일수", fontsize=8)
frame(bx, "달력의 날 종류별 구성")
fig.tight_layout()
fig.savefig("fig_15_7.png", dpi=300)
plt.show()

## 15.3 주성분 분석

In [ ]:
sub = pd.read_csv([f for f in csvs if "사용월" in first_line(f)][0], encoding="cp949")
month = sub[sub["사용월"] == 202510].copy()
ON, OFF = [c for c in sub.columns if "승차" in c], [c for c in sub.columns if "하차" in c]
month = month[month[ON + OFF].sum(axis=1) > 100000].reset_index(drop=True)      # 한 달 승하차가 10만 명이 안 되는 역은 뺀다
names = (month["지하철역"] + "(" + month["호선명"] + ")").to_numpy()
P = month[ON + OFF].to_numpy(float)
P = P / P.sum(axis=1, keepdims=True)                                            # 역마다 한 달 승하차 전체를 1로 두고 48개 칸의 비율로 바꾼다
HOURS = [int(c[:2]) for c in ON]
print(P.shape, ON[0], ON[-1], OFF[0])
for who in ["역삼(2호선)", "상계(4호선)"]:
    i = list(names).index(who)
    print(who, (P[i, [3, 4, 14]] * 100).round(1).tolist(), (P[i, [27, 28, 38]] * 100).round(1).tolist())

In [ ]:
# 그림 15-8
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True)
xs = np.arange(24)
for ax, who in zip(axes, ["역삼(2호선)", "상계(4호선)"]):
    i = list(names).index(who)
    ax.bar(xs - 0.2, P[i, :24] * 100, 0.4, color=NAVY, label="승차")
    ax.bar(xs + 0.2, P[i, 24:] * 100, 0.4, color=BRICK, label="하차")
    ax.set_xticks(xs[::3], [f"{HOURS[k]}시" for k in xs[::3]], fontsize=7)
    ax.set_xlabel("시간대(시작 시각)", fontsize=8)
    ax.legend(frameon=False, fontsize=7.5, loc="upper center", ncol=2)
    frame(ax, who)
axes[0].set_ylabel("한 달 승하차 전체에 대한 비율(%)", fontsize=8)
axes[0].set_ylim(0, 14.5)
fig.tight_layout()
fig.savefig("fig_15_8.png", dpi=300)
plt.show()

In [ ]:
middle = P.mean(axis=0)
Z = P - middle                                                                   # 역마다 평균적인 역과의 차이
U, S, Vt = np.linalg.svd(Z, full_matrices=False)                                 # Vt의 행 하나가 축 하나다. 분산이 큰 축부터 나온다
part = S ** 2 / (S ** 2).sum()                                                   # 축마다 설명하는 분산의 비율
print(part[:6].round(3).tolist(), part.cumsum()[[0, 1, 2, 4, 9]].round(3).tolist())
place = Z @ Vt.T                                                                 # 역마다 새 축에서의 좌표. 열이 축

cov = Z.T @ Z / (len(Z) - 1)                                                     # 48 × 48 공분산 행렬
print(np.allclose(cov @ Vt[0], S[0] ** 2 / (len(Z) - 1) * Vt[0]), round(float(place[:, 0].var(ddof=1) / np.trace(cov)), 3))

from sklearn.decomposition import PCA
print(PCA(3).fit(P).explained_variance_ratio_.round(3).tolist())

load1 = pd.DataFrame({"승차": Vt[0, :24], "하차": Vt[0, 24:]}, index=[f"{h_}시" for h_ in HOURS])
print(load1.loc[["6시", "7시", "8시", "9시", "12시", "17시", "18시", "19시", "21시"]].round(2).T.to_string())
order = np.argsort(place[:, 0])
print(names[order[:6]].tolist())
print(names[order[-6:]].tolist())

In [ ]:
# 그림 15-9
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 3.3))
n_ax = np.arange(1, 13)
ax.bar(n_ax, part[:12] * 100, color=[BRICK, TEAL, "#8a6d3b"] + [NAVY] * 9, width=0.65)
for x, v in zip(n_ax[:5], part[:5] * 100):
    ax.text(x, v + 1.5, f"{v:.1f}", fontsize=7, ha="center", color=NAVY)
ax.set_xticks(n_ax)
ax.set_ylim(0, 76)
ax.set_xlabel("주성분의 번호", fontsize=8)
ax.set_ylabel("설명하는 분산의 비율(%)", fontsize=8)
frame(ax, "주성분마다의 비율")
bx.plot(n_ax, part.cumsum()[:12] * 100, color=NAVY, lw=1.8, marker="o", ms=3.5)
for x in (1, 2, 3, 5, 10):
    bx.text(x + 0.25, part.cumsum()[x - 1] * 100 - 3.2, f"{part.cumsum()[x - 1] * 100:.1f}", fontsize=7, color=NAVY)
bx.axhline(90, color=GRAY, lw=0.8, ls=":")
bx.text(12.3, 90.6, "90%", fontsize=7, color=GRAY, ha="right", va="bottom")
bx.set_xticks(n_ax)
bx.set_ylim(60, 101)
bx.set_xlabel("앞에서부터 쓴 주성분의 수", fontsize=8)
bx.set_ylabel("누적 비율(%)", fontsize=8)
frame(bx, "누적 비율")
fig.tight_layout()
fig.savefig("fig_15_9.png", dpi=300)
plt.show()

In [ ]:
# 그림 15-10
fig, axes = plt.subplots(3, 1, figsize=(7.2, 6.6), sharex=True)
for a, (ax, color) in enumerate(zip(axes, [BRICK, TEAL, "#8a6d3b"])):
    ax.bar(xs - 0.2, Vt[a, :24], 0.4, color=NAVY, label="승차")
    ax.bar(xs + 0.2, Vt[a, 24:], 0.4, color=BRICK, label="하차")
    ax.axhline(0, color="black", lw=0.6)
    ax.set_ylabel("적재값", fontsize=8)
    ax.set_ylim(-0.62, 0.55)
    frame(ax, f"주성분 {a + 1} (분산의 {part[a] * 100:.1f}%)")
    if a == 0:
        ax.legend(frameon=False, fontsize=7.5, loc="upper right", ncol=2)
axes[2].set_xticks(xs, [f"{h_}" for h_ in HOURS], fontsize=7)
axes[2].set_xlabel("시간대(시작 시각)", fontsize=8)
[a_.tick_params(labelbottom=True) for a_ in fig.axes]
fig.tight_layout()
fig.savefig("fig_15_10.png", dpi=300)
plt.show()

In [ ]:
for axis in [1, 2]:
    order = np.argsort(place[:, axis])
    print(axis + 1, names[order[:5]].tolist())
    print(axis + 1, names[order[-5:]].tolist())
back = [round(float(((Z - place[:, :k] @ Vt[:k]) ** 2).sum() / (Z ** 2).sum()), 3) for k in (1, 2, 3, 5, 10)]   # 주성분 k개로 되돌렸을 때 남는 오차의 비율
print(back)
i = list(names).index("역삼(2호선)")
print(place[i, :3].round(3).tolist(), [round(float(np.abs(P[i] - (middle + place[i, :k] @ Vt[:k])).max()) * 100, 2) for k in (0, 1, 2, 3)])
scaled = Z / P.std(axis=0)                                                       # 열마다 표준편차로 나눈 뒤에 주성분을 구하면
U2, S2, Vt2 = np.linalg.svd(scaled, full_matrices=False)
print((S2 ** 2 / (S2 ** 2).sum())[:3].round(3).tolist(), round(float(abs(np.corrcoef(place[:, 0], scaled @ Vt2[0])[0, 1])), 3),
      round(float(abs(np.corrcoef(place[:, 1], scaled @ Vt2[1])[0, 1])), 3))
print((P.std(axis=0)[[4, 28, 21, 22]] * 100).round(3).tolist())               # 8시 승차, 8시 하차, 1시 승차, 2시 승차의 표준편차(%)

In [ ]:
# 그림 15-11
fig, ax = plt.subplots(figsize=(7.2, 5.6))
ax.scatter(place[:, 0] * 100, place[:, 1] * 100, s=9, color=GRAY, alpha=0.75, lw=0)
picked = ["역삼(2호선)", "가산디지털단지(7호선)", "여의도(5호선)", "상계(4호선)", "홍대입구(2호선)", "종로3가(1호선)", "대공원(과천선)", "강매(경의선)", "소요산(경원선)", "잠실(송파구청)(2호선)"]
offsets = {}
for who in picked:
    if who not in list(names):
        continue
    i = list(names).index(who)
    x, yv = place[i, 0] * 100, place[i, 1] * 100
    ax.scatter(x, yv, s=26, color=BRICK, zorder=4, lw=0)
    dx, dy, ha = {"종로3가(1호선)": (-0.5, 0.5, "right"), "대공원(과천선)": (0.5, 0.5, "left"), "역삼(2호선)": (-0.5, 0.5, "right"), "여의도(5호선)": (-0.5, 0.5, "right"),
                  "가산디지털단지(7호선)": (-0.5, -1.1, "right"), "홍대입구(2호선)": (-0.5, 0.5, "right")}.get(who, (0.5, 0.45, "left"))
    ax.text(x + dx, yv + dy, who, fontsize=7, color=NAVY, ha=ha, bbox=BOX, zorder=5)
ax.axhline(0, color=NAVY, lw=0.5); ax.axvline(0, color=NAVY, lw=0.5)
ax.set_xlabel("주성분 1의 좌표(× 100). 오른쪽일수록 아침에 내리고 저녁에 타는 역", fontsize=8)
ax.set_ylabel("주성분 2의 좌표(× 100). 위쪽일수록 낮 시간대의 이용이 많은 역", fontsize=8)
frame(ax, "역 567곳을 두 주성분의 좌표로 나타낸 것", grid="both")
fig.tight_layout()
fig.savefig("fig_15_11.png", dpi=300)
plt.show()

## 15.4 레이블 없이 결과를 확인하는 방법

In [ ]:
old = sub[sub["사용월"] == 202310].copy()                                        # 2년 전 같은 달
old.index = old["지하철역"] + "(" + old["호선명"] + ")"
old = old.reindex(names).dropna(subset=[ON[0]])
P_old = old[ON + OFF].to_numpy(float)
P_old = P_old / P_old.sum(axis=1, keepdims=True)
kept = np.isin(names, old.index)
then, now = (P_old - middle) @ Vt[:3].T, place[kept][:, :3]                      # 2025년에 구한 축에 2023년의 역을 놓는다
print(int(kept.sum()), [round(float(np.corrcoef(then[:, a], now[:, a])[0, 1]), 3) for a in range(3)])
Vt_old = np.linalg.svd(P_old - P_old.mean(axis=0), full_matrices=False)[2]       # 2023년 자료로 축을 따로 구한다
print([round(float(abs(Vt_old[a] @ Vt[a])), 3) for a in range(5)])

rng = np.random.default_rng(0)
rows = {2: [], 4: []}
for k, ref in [(2, two), (4, four)]:
    for _ in range(20):
        again = rng.integers(0, len(shape), len(shape))                          # 날을 복원 추출해서 처음부터 다시 군집을 찾는다
        centers = best_of(shape[again], k, tries=5)[1]
        relabel = ((shape[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
        rows[k].append(adjusted_rand_score(ref, relabel))
    print(k, round(float(np.min(rows[k])), 2), round(float(np.mean(rows[k])), 2))

In [ ]:
# 그림 15-12
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 3.6))
ax.scatter(then[:, 0] * 100, now[:, 0] * 100, s=7, color=NAVY, alpha=0.6, lw=0)
lim = [min(then[:, 0].min(), now[:, 0].min()) * 100 - 1, max(then[:, 0].max(), now[:, 0].max()) * 100 + 1]
ax.plot(lim, lim, color=GRAY, lw=0.9, ls="--")
ax.text(lim[0] + 1, lim[1] - 3.2, f"역 {int(kept.sum())}곳\n상관계수 {np.corrcoef(then[:, 0], now[:, 0])[0, 1]:.3f}", fontsize=7.5, color=NAVY, bbox=BOX)
ax.set_xlabel("2023년 10월의 주성분 1 좌표(× 100)", fontsize=8)
ax.set_ylabel("2025년 10월의 주성분 1 좌표(× 100)", fontsize=8)
frame(ax, "같은 역의 2년 전 좌표", grid="both")
bins = np.linspace(0.4, 1.0, 13)
bx.hist([rows[2], rows[4]], bins=bins, color=[NAVY, BRICK], label=["군집 2개", "군집 4개"])
bx.set_xlabel("원래 군집과의 일치(조정 랜드 지수)", fontsize=8)
bx.set_ylabel("복원 추출 20번 가운데 횟수", fontsize=8)
bx.set_yticks(range(0, 21, 4))
bx.legend(frameon=False, fontsize=7.5, loc="upper left")
frame(bx, "날을 다시 뽑아 군집을 찾았을 때")
fig.tight_layout()
fig.savefig("fig_15_12.png", dpi=300)
plt.show()

In [ ]:
like_weekend = four == 3                                                         # 군집 라
strange = like_weekend & (kind == "평일")                                        # 달력으로는 평일인데 군집 라에 속한 날
print(int(strange.sum()), pd.Series(dates[strange].strftime("%Y-%m")).value_counts().sort_index().to_dict())
print([d_ for d_ in dates[strange].strftime("%Y-%m-%d") if d_.endswith("05-01")])
noon = pd.DataFrame({"연": dates.year, "월": dates.month, "13시": shape[:, 12]})[kind == "평일"]
print(noon[noon["월"].isin([1, 4, 8])].groupby(["월", "연"])["13시"].mean().unstack().round(3).to_string())

In [ ]:
weather = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "일사(MJ/m2)"]) for f in csvs if first_line(f).startswith("지점,지점명")])
weather = weather[weather["지점명"].isin(["서울", "대전", "대구", "부산", "광주"])]
weather["날"] = (pd.to_datetime(weather["일시"]) - pd.Timedelta(hours=1)).dt.normalize()
sunshine = weather.groupby(["날", "지점명"])["일사(MJ/m2)"].sum().unstack().mean(axis=1)        # 다섯 도시의 하루 일사량 평균

spring = (dates >= "2025-03-10") & (dates <= "2025-05-09") & (kind == "평일")
inside, outside = sunshine.reindex(dates[spring & strange]), sunshine.reindex(dates[spring & ~strange])
print(len(inside), round(float(inside.mean()), 1), round(float(inside.min()), 1))
print(len(outside), round(float(outside.mean()), 1), outside.quantile([0.25, 0.5, 0.75]).round(1).tolist())
sun_all = sunshine.reindex(dates[spring]).to_numpy()
noon_all = shape[spring, 12]
print(round(float(np.corrcoef(sun_all, noon_all)[0, 1]), 2))
for yr in (2023, 2025):
    sp = (dates >= f"{yr}-03-10") & (dates <= f"{yr}-05-09") & (kind == "평일")
    s_, n_ = sunshine.reindex(dates[sp]).to_numpy(), shape[sp, 12]
    ok = ~np.isnan(s_)
    slope = np.polyfit(s_[ok], n_[ok], 1)
    print(yr, int(ok.sum()), round(float(np.corrcoef(s_[ok], n_[ok])[0, 1]), 2), round(float(slope[0]) * 10, 3))

In [ ]:
# 그림 15-13
fig, ax = plt.subplots(figsize=(7.2, 4.2))
is_odd = strange[spring]
ax.scatter(sun_all[~is_odd], noon_all[~is_odd], s=22, color=NAVY, alpha=0.8, lw=0, label=f"다른 군집에 속한 평일 {int((~is_odd).sum())}일")
ax.scatter(sun_all[is_odd], noon_all[is_odd], s=30, color=BRICK, lw=0, label=f"군집 라에 속한 평일 {int(is_odd.sum())}일")
fit = np.polyfit(sun_all, noon_all, 1)
gx = np.array([sun_all.min(), sun_all.max()])
ax.plot(gx, np.polyval(fit, gx), color=GRAY, lw=1, ls="--", label=f"직선 맞춤(상관계수 {np.corrcoef(sun_all, noon_all)[0, 1]:.2f})")
ax.axhline(1, color=GRAY, lw=0.7, ls=":")
ax.set_xlabel("다섯 도시의 하루 일사량 평균(MJ/m²)", fontsize=8)
ax.set_ylabel("13시의 값(그날의 평균에 대한 비)", fontsize=8)
ax.legend(frameon=False, fontsize=7.5, loc="lower left")
frame(ax, "2025년 3월 10일~5월 9일의 평일", grid="both")
fig.tight_layout()
fig.savefig("fig_15_13.png", dpi=300)
plt.show()